# 04 — SFT Training

Fine-tune GPT-2 on a tiny instruction dataset using `SFTTrainer`.

This notebook is intentionally small (10 examples, 2 epochs) so it runs in ~1 minute on Colab CPU.

> **Colab users:** Run the setup cell, restart runtime, then continue.

In [1]:
import subprocess, sys, os

def _is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _is_colab():
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install',
         'git+https://github.com/silvaxxx1/MyLLM.git'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print('Install FAILED:', r.stderr[-2000:])
    else:
        print('Done! Now go to Runtime → Restart runtime, then continue.')
else:
    root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    has_uv = subprocess.run(['which', 'uv'], capture_output=True).returncode == 0
    cmd = ['uv', 'pip', 'install', '-e', root] if has_uv else [sys.executable, '-m', 'pip', 'install', '-e', root]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('Install FAILED:', r.stderr[-2000:])
    else:
        print('Local editable install ready.')


Done! Now go to Runtime → Restart runtime, then continue.


## 1. Imports

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer as HFTokenizer

from myllm import ModelConfig
from myllm.train import SFTTrainer, SFTTrainerConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch :', torch.__version__)
print('device:', device)

torch : 2.10.0+cu128
device: cuda


## 2. Tiny instruction dataset

In [3]:
EXAMPLES = [
    {'instruction': 'What is the capital of France?',      'response': 'The capital of France is Paris.'},
    {'instruction': 'What is 2 + 2?',                     'response': '2 + 2 equals 4.'},
    {'instruction': 'Name a planet in our solar system.',  'response': 'Earth is a planet in our solar system.'},
    {'instruction': 'What color is the sky?',              'response': 'The sky is blue during the day.'},
    {'instruction': 'What is machine learning?',           'response': 'Machine learning is a subfield of AI where models learn from data.'},
    {'instruction': 'Who wrote Romeo and Juliet?',         'response': 'Romeo and Juliet was written by William Shakespeare.'},
    {'instruction': 'What is the speed of light?',         'response': 'The speed of light is approximately 299,792 km/s.'},
    {'instruction': 'What does DNA stand for?',            'response': 'DNA stands for Deoxyribonucleic Acid.'},
    {'instruction': 'What is a neural network?',           'response': 'A neural network is a computational model inspired by the human brain.'},
    {'instruction': 'What year did World War II end?',     'response': 'World War II ended in 1945.'},
]

TEMPLATE = '### Instruction:\n{instruction}\n\n### Response:\n{response}'

print('Sample formatted example:')
print(TEMPLATE.format(**EXAMPLES[0]))

Sample formatted example:
### Instruction:
What is the capital of France?

### Response:
The capital of France is Paris.


## 3. PyTorch Dataset

In [4]:
class InstructionDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length=128):
        self.items = []
        for ex in examples:
            enc = tokenizer(
                TEMPLATE.format(**ex),
                max_length=max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt',
            )
            ids = enc['input_ids'].squeeze(0)
            self.items.append({
                'input_ids':      ids,
                'attention_mask': enc['attention_mask'].squeeze(0),
                'labels':         ids.clone(),
            })

    def __len__(self):          return len(self.items)
    def __getitem__(self, i):   return self.items[i]


tokenizer = HFTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

dataset    = InstructionDataset(EXAMPLES, tokenizer)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

print(f'Dataset : {len(dataset)} examples  |  {len(dataloader)} batches/epoch')
print(f'Shape   : {next(iter(dataloader))["input_ids"].shape}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Dataset : 10 examples  |  5 batches/epoch
Shape   : torch.Size([2, 128])


## 4. Configure the trainer

In [5]:
trainer_cfg = SFTTrainerConfig(
    output_dir='./demo_sft_output',
    num_epochs=2,
    batch_size=2,
    gradient_accumulation_steps=1,
    max_grad_norm=1.0,
    logging_steps=1,
    save_steps=0,       # no checkpoints in this demo
    report_to=[],       # no WandB
    model_config_name='gpt2-small',
)

print('SFTTrainerConfig ready.')
print(f'  output_dir = {trainer_cfg.output_dir}')
print(f'  num_epochs = {trainer_cfg.num_epochs}')

SFTTrainerConfig ready.
  output_dir = ./demo_sft_output
  num_epochs = 2


## 5. Train

In [6]:
model_cfg = ModelConfig.from_name('gpt2-small')

trainer = SFTTrainer(trainer_cfg, model_config=model_cfg)
trainer.setup_model()
trainer.setup_data(train_dataloader=dataloader)
trainer.setup_optimizer()

trainer.train()

─────────────────────────────────────── Starting SFT Training for 2 epochs ────────────────────────────────────────

Output()

/usr/local/lib/python3.12/dist-packages/myllm/Train/base_trainer.py:203: UserWarning: Detected call of 
`lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite 
order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the 
first value of the learning rate schedule. See more details at 
https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.scheduler.step()

Output()

                    SFT Training Summary                    
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Epochs ┃ Final Eval Loss ┃ Best Checkpoint ┃ W&B Run URL ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│   2    │       N/A       │       N/A       │     N/A     │
└────────┴─────────────────┴─────────────────┴─────────────┘

───────────────────────────────────────────── SFT Training Completed! ─────────────────────────────────────────────

## 6. Generate from the fine-tuned model

In [7]:
from myllm import LLM, GenerationConfig

# Wrap trained model in LLM for generation
llm = LLM(config=model_cfg, device=str(trainer.device))
llm.model = trainer.model
llm.model.eval()

gen_cfg = GenerationConfig(
    max_length=40, temperature=0.7,
    top_k=50, do_sample=True,
    use_kv_cache=True, use_optimized_sampler=True,
)

test_prompts = [
    '### Instruction:\nWhat is the capital of France?\n\n### Response:\n',
    '### Instruction:\nWhat is 2 + 2?\n\n### Response:\n',
]

for prompt in test_prompts:
    result = llm.generate_text(prompt, tokenizer, gen_cfg)
    print(result['text'])
    print('-' * 60)


### Instruction:
What is the capital of France?

### Response:

------------------------------------------------------------
### Instruction:
What is 2 + 2?

### Response:
Adams
------------------------------------------------------------


## 7. Save the fine-tuned weights

In [8]:
import os

save_path = './demo_sft_output/finetuned_gpt2.pt'
llm.save(save_path)
print('Saved to:', os.path.abspath(save_path))

✅ Model saved to ./demo_sft_output/finetuned_gpt2.pt
Saved to: /content/demo_sft_output/finetuned_gpt2.pt
